In [1]:
# If not installed already (uncomment as needed):
# !pip install torch torch-geometric pandas tqdm

import os, json, torch, pandas as pd
from pathlib import Path

from data_utils import load_graphs, attach_labels_and_noisy, build_loaders
from train_loop import train, evaluate


In [2]:
# EDIT THESE:
GRAPHS_PT = "../processed/graphs_train.pt"      # <-- Step 1 output (a list[Data])
LABELS_CSV = "labels_train.csv"    # <-- We'll generate a template first if it doesn't exist
MODEL_OUT  = "qem_graph_transformer.pt"


In [3]:
graphs = load_graphs(GRAPHS_PT)
print(f"Loaded {len(graphs)} graphs")
# quick sanity on shapes
g0 = graphs[0]
print("x:", tuple(g0.x.shape),
      "edge_index:", tuple(g0.edge_index.shape),
      "lightcone_masks:", tuple(g0.lightcone_masks.shape))
M = g0.lightcone_masks.shape[1]
print("Measured qubits (M):", M)


Loaded 500 graphs
x: (2071, 20) edge_index: (2, 2565) lightcone_masks: (2071, 5)
Measured qubits (M): 5


In [4]:
# This will create a template with circuit_path + zero arrays for noisy_z/target_y (len = M).
# If LABELS_CSV already exists, this cell won’t overwrite it unless you set FORCE=True.
FORCE = False

if (not Path(LABELS_CSV).exists()) or FORCE:
    rows = []
    for g in graphs:
        path = g.circuit_path if isinstance(g.circuit_path, str) else g.circuit_path[0]
        # Placeholder arrays (fill these with your real noisy/ideal values):
        zeros = [0.0] * g.lightcone_masks.shape[1]
        rows.append({
            "circuit_path": path,
            "noisy_z_json": json.dumps(zeros),
            "target_y_json": json.dumps(zeros)
        })
    df = pd.DataFrame(rows)
    df.to_csv(LABELS_CSV, index=False)
    print(f"Template written to {LABELS_CSV} with {len(df)} rows and M={M}.")
    display(df.head(3))
else:
    print(f"{LABELS_CSV} already exists — not overwriting. Set FORCE=True to regenerate.")


Template written to labels_train.csv with 500 rows and M=5.


,circuit_path,noisy_z_json,target_y_json
0,../../tutorials/data/ising_zne_hardware/100q_b...,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0]"
1,../../tutorials/data/ising_zne_hardware/100q_b...,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0]"
2,../../tutorials/data/ising_zne_hardware/100q_b...,"[0.0, 0.0, 0.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0]"
